In [ ]:
# ══════════════════════════════════════════════
# PRELUDE — run this cell first
# Added by fix pass. Everything below your own code is unchanged.
# ══════════════════════════════════════════════

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timezone

class Stale(RuntimeError): pass
class Unconverged(RuntimeError): pass
class TooFewObs(RuntimeError): pass


def safe_at(obj, i=-1, col=None):
    """float() on a 1-element Series is deprecated. Handles yfinance MultiIndex columns."""
    s = obj[col] if col is not None else obj
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    a = np.asarray(s.dropna()).ravel()
    if a.size == 0:
        raise Stale("empty series")
    return float(a[i])


def safe_last(obj, col=None):
    return safe_at(obj, -1, col)


def unmute_convergence():
    """Let convergence failures through. They were being swallowed by filterwarnings('ignore')."""
    try:
        import statsmodels.tools.sm_exceptions as _s
        for _n in ("ConvergenceWarning", "EstimationWarning", "ValueWarning"):
            if hasattr(_s, _n):
                warnings.filterwarnings("always", category=getattr(_s, _n))
    except Exception:
        pass
    try:
        from arch.utility.exceptions import DataScaleWarning
        warnings.filterwarnings("always", category=DataScaleWarning)
    except Exception:
        pass


# ── CONTRACT: pin which gold series this notebook uses ──────────────
#   'front_intraday' = GC=F 30-min  (the contract you trade — DEFAULT)
#   'front_daily'    = GC=F daily   (often prints spot, not the future)
#   'spot'           = XAUUSD
LIVE_CONTRACT = "front_intraday"


def get_ctx(contract=LIVE_CONTRACT):
    if contract == "front_intraday":
        gc_df = yf.download("GC=F", period="5d", interval="30m", progress=False)
    elif contract == "front_daily":
        gc_df = yf.download("GC=F", period="1mo", interval="1d", progress=False)
    elif contract == "spot":
        gc_df = yf.download("XAUUSD=X", period="5d", interval="30m", progress=False)
    else:
        raise ValueError(contract)

    gc = safe_last(gc_df, "Close")
    gld = safe_last(yf.download("GLD", period="5d", progress=False), "Close")

    def _s(t, d):
        try:
            return safe_last(yf.download(t, period="5d", progress=False), "Close")
        except Exception:
            return d

    if not 1000 < gc < 12000:
        raise Stale(f"GC={gc:,.2f} implausible — bad fetch")
    if not 100 < gld < 1200:
        raise Stale(f"GLD={gld:,.2f} implausible — bad fetch")
    ratio = gc / gld
    if not 9.5 <= ratio <= 12.5:
        raise Stale(f"GLD->gold ratio {ratio:.3f}x outside [9.5, 12.5] — "
                    f"prices are from different dates")

    # known failure point: GC=F daily and 30m disagree
    try:
        _d = safe_last(yf.download("GC=F", period="1mo", interval="1d", progress=False), "Close")
        _i = safe_last(yf.download("GC=F", period="5d", interval="30m", progress=False), "Close")
        if abs(_d - _i) / _i > 0.005:
            print(f"  !! GC=F daily {_d:,.2f} vs 30m {_i:,.2f} — ${abs(_d-_i):,.1f} apart.")
            print(f"     Using '{contract}'. VERIFY THE SETTLE IN QUANTOWER.")
    except Exception:
        pass

    ctx = dict(gc=gc, gld=gld, ratio=ratio, contract=contract,
               asof=str(gc_df.index[-1]),
               vix=_s("^VIX", np.nan), gvz=_s("^GVZ", np.nan), rf=_s("^IRX", 4.0) / 100)
    print(f"  contract : {contract}")
    print(f"  GC {gc:>10,.2f}   GLD {gld:>8,.2f}   ratio {ratio:.4f}x   <-- NOT 10.0")
    print(f"  VIX {ctx['vix']:.2f}   GVZ {ctx['gvz']:.1f}   rf {ctx['rf']:.2%}   as of {ctx['asof']}")
    return ctx


def need_obs(n, floor, label=""):
    if n < floor:
        raise TooFewObs(f"{label}: {n} observations, need >= {floor}. Do not report this fit.")


def need_fresh(as_of, days=7, label="field"):
    age = (datetime.now(timezone.utc).date() - datetime.fromisoformat(as_of).date()).days
    if age > days:
        raise Stale(f"{label} written {as_of} ({age}d ago) — EXPIRED. Rewrite or delete it.")
    return True


def need_converged(res, states=None, label="model"):
    conv = getattr(res, "converged", None)
    if conv is None and isinstance(getattr(res, "mle_retvals", None), dict):
        conv = res.mle_retvals.get("converged", True)
    if conv is False:
        raise Unconverged(f"{label}: optimiser did not converge. Not a regime classification.")
    if states is not None:
        u, c = np.unique(np.asarray(states), return_counts=True)
        if len(u) < 2:
            raise Unconverged(f"{label}: {len(u)} distinct state — a flat line, not regimes.")
        if c.min() / c.sum() < 0.05:
            raise Unconverged(f"{label}: minority state is {c.min()/c.sum():.1%} — "
                              f"outlier detector, not a regime model.")


def kelly_cap(f, frac=0.25, cap=0.20):
    out = float(np.clip(f * frac, -cap, cap))
    if abs(f) > 1:
        print(f"  ! raw f*={f:.2f} implies {f*100:.0f}% of capital (small-sample artefact). "
              f"Using {out:.1%}.")
    return out


LIVE_CTX   = get_ctx()
LIVE_RATIO = LIVE_CTX["ratio"]   # use instead of 10
LIVE_GC    = LIVE_CTX["gc"]

# NOTE: named LIVE_* on purpose — GOLD is your hex colour in 15 cells,
#       and RATIO / CONTRACT are already used in cells 44-45.


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# COT AUTO-FETCH — CFTC Disaggregated, Futures-and-Options COMBINED
# Source: https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm
#
# NOTE: your old code pulled f_year.txt from fut_disagg_* = FUTURES ONLY,
#       while every dashboard header said "Options and Futures Combined".
#       This uses com_disagg_* -> c_year.txt = actually COMBINED.
#
# The annual zip is rewritten by CFTC every Friday ~15:30 ET, so pulling
# the current year's file always gives the newest report. No manual step.
# ══════════════════════════════════════════════════════════════════════

import io, os, zipfile, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path

COT_CACHE = Path.home() / ".cot_cache"
COT_CACHE.mkdir(exist_ok=True)

# Disaggregated Futures-and-Options Combined, annual history
COT_HIST_URL = "https://www.cftc.gov/files/dea/history/com_disagg_txt_{year}.zip"
# Current-week snapshot (headerless; used only as a freshness cross-check)
COT_CURRENT_URL = "https://www.cftc.gov/dea/newcot/c_disagg.txt"

CFTC_CODES = {
    "gold":        "088691",
    "silver":      "084691",
    "copper":      "085692",
    "wti_crude":   "067651",
    "natgas":      "023651",
    "corn":        "002602",
    "platinum":    "076651",
    "palladium":   "075651",
}

_COT_UA = {"User-Agent": "Mozilla/5.0 (research; contact: elena)"}


def _cot_expected_report_date(now=None):
    """
    COT is Tuesday data released Friday 15:30 ET. Returns the latest
    Tuesday that should be published by now.
    """
    now = now or datetime.now(timezone.utc)
    et = now - timedelta(hours=4)                    # ET approx (EDT)
    days_since_fri = (et.weekday() - 4) % 7
    last_fri = (et - timedelta(days=days_since_fri)).replace(
        hour=15, minute=30, second=0, microsecond=0)
    if et < last_fri:
        last_fri -= timedelta(days=7)
    return (last_fri - timedelta(days=3)).date()     # the Tuesday it covers


def _cot_download_year(year, force=False):
    """Download and cache one year of combined disaggregated data."""
    cache = COT_CACHE / f"com_disagg_{year}.parquet"
    stamp = COT_CACHE / f"com_disagg_{year}.stamp"

    # current year: refresh if cache older than 12h; past years never change
    fresh = False
    if cache.exists() and not force:
        if year < datetime.now().year:
            fresh = True
        elif stamp.exists():
            age_h = (datetime.now().timestamp() - float(stamp.read_text())) / 3600
            fresh = age_h < 12
    if fresh:
        return pd.read_parquet(cache)

    url = COT_HIST_URL.format(year=year)
    r = requests.get(url, timeout=60, headers=_COT_UA)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        names = z.namelist()
        inner = next((n for n in names if n.lower().endswith((".txt", ".csv"))), None)
        if inner is None:
            raise RuntimeError(f"{year}: no txt/csv in zip — got {names}")
        if inner.lower().startswith("f_"):
            raise RuntimeError(
                f"{year}: zip contains '{inner}' (f_ = FUTURES ONLY). "
                f"Expected 'c_year.txt' from com_disagg_*. Wrong URL.")
        with z.open(inner) as fh:
            df = pd.read_csv(fh, low_memory=False)

    df.columns = [c.strip() for c in df.columns]
    cache.parent.mkdir(exist_ok=True)
    df.to_parquet(cache, index=False)
    stamp.write_text(str(datetime.now().timestamp()))
    print(f"    downloaded {year}  ({inner}, {len(df):,} rows)")
    return df


def _cot_date_col(df):
    for c in ("Report_Date_as_YYYY-MM-DD", "Report_Date_as_MM_DD_YYYY",
              "As_of_Date_In_Form_YYMMDD", "Report_Date"):
        if c in df.columns:
            return c
    raise KeyError(f"no date column found in {list(df.columns)[:12]}")


def _cot_parse_dates(s, col):
    if "YYMMDD" in col:
        return pd.to_datetime(s.astype(str).str.zfill(6), format="%y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")


def fetch_cot(market="gold", years=3, force=False, verbose=True):
    """
    Returns a tidy weekly DataFrame for one market, newest last:

        date, open_interest,
        prod_long, prod_short, prod_net,
        swap_long, swap_short, swap_net,
        mm_long,   mm_short,   mm_net,
        other_net, comm_net (prod+swap),
        mm_index, comm_index   (156-week rolling 0-100 Williams index)

    Raises on anything implausible rather than returning zeros.
    """
    code = CFTC_CODES.get(market.lower(), market)
    this_year = datetime.now().year
    frames = []
    if verbose:
        print(f"  CFTC Disaggregated · Futures-and-Options COMBINED · code {code}")

    for y in range(this_year - years + 1, this_year + 1):
        try:
            frames.append(_cot_download_year(y, force=force))
        except Exception as e:
            print(f"    ! {y}: {type(e).__name__}: {e}")
    if not frames:
        raise RuntimeError("no COT data retrieved — check network / CFTC availability")

    raw = pd.concat(frames, ignore_index=True)

    code_col = next((c for c in ("CFTC_Contract_Market_Code", "CFTC_Contract_Market_Code_Quotes")
                     if c in raw.columns), None)
    if code_col is None:
        raise KeyError("no CFTC_Contract_Market_Code column")

    m = raw[raw[code_col].astype(str).str.strip().str.zfill(6) == code].copy()
    if m.empty:
        names = raw["Market_and_Exchange_Names"].dropna().unique()[:5]
        raise ValueError(f"code {code} not found. Sample markets: {list(names)}")

    dcol = _cot_date_col(m)
    m["date"] = _cot_parse_dates(m[dcol], dcol)
    m = m.dropna(subset=["date"]).sort_values("date")
    m = m.drop_duplicates(subset=["date"], keep="last")

    def col(*cands):
        for c in cands:
            if c in m.columns:
                return pd.to_numeric(m[c], errors="coerce")
        raise KeyError(f"none of {cands} present")

    out = pd.DataFrame({
        "date":          m["date"].values,
        "open_interest": col("Open_Interest_All"),
        "prod_long":     col("Prod_Merc_Positions_Long_All"),
        "prod_short":    col("Prod_Merc_Positions_Short_All"),
        "swap_long":     col("Swap_Positions_Long_All"),
        "swap_short":    col("Swap__Positions_Short_All", "Swap_Positions_Short_All"),
        "mm_long":       col("M_Money_Positions_Long_All"),
        "mm_short":      col("M_Money_Positions_Short_All"),
        "other_long":    col("Other_Rept_Positions_Long_All"),
        "other_short":   col("Other_Rept_Positions_Short_All"),
    }).reset_index(drop=True)

    out["prod_net"]  = out.prod_long  - out.prod_short
    out["swap_net"]  = out.swap_long  - out.swap_short
    out["mm_net"]    = out.mm_long    - out.mm_short
    out["other_net"] = out.other_long - out.other_short
    out["comm_net"]  = out.prod_net + out.swap_net     # producers + swap dealers

    w = min(156, len(out))
    for c in ("mm_net", "comm_net", "swap_net"):
        lo = out[c].rolling(w, min_periods=20).min()
        hi = out[c].rolling(w, min_periods=20).max()
        out[c.replace("_net", "_index")] = 100 * (out[c] - lo) / (hi - lo).replace(0, np.nan)

    # ---- validation: fail loud rather than score a broken parse ----------
    last = out.iloc[-1]
    if last.open_interest < 10_000:
        raise ValueError(f"open interest {last.open_interest:,.0f} implausible — bad parse")
    for f in ("prod_net", "swap_net", "mm_net"):
        if last[f] == 0:
            raise ValueError(f"{f} parsed as exactly 0 — column mismatch, not flat positioning")

    expected = _cot_expected_report_date()
    lag = (expected - last.date.date()).days
    if lag > 7:
        print(f"    !! newest report {last.date.date()} but {expected} expected "
              f"({lag}d stale). CFTC may be delayed, or cache is stale — force=True to refresh.")
    elif verbose:
        print(f"    latest report : {last.date.date()}  (current)")

    if verbose:
        print(f"    rows          : {len(out)}  [{out.date.min().date()} -> {out.date.max().date()}]")
        print(f"    OI            : {last.open_interest:>10,.0f}")
        print(f"    Managed Money : {last.mm_net:>+10,.0f}   index {last.mm_index:5.1f}/100")
        print(f"    Swap Dealers  : {last.swap_net:>+10,.0f}   index {last.swap_index:5.1f}/100")
        print(f"    Prod/Merch    : {last.prod_net:>+10,.0f}")
        print(f"    Commercial    : {last.comm_net:>+10,.0f}   index {last.comm_index:5.1f}/100")

    return out


def cot_signal(df, extreme_lo=20, extreme_hi=80):
    """Williams rule: need BOTH commercial and managed-money at an extreme."""
    r = df.iloc[-1]
    c, m = r.comm_index, r.mm_index
    if np.isnan(c) or np.isnan(m):
        return "INSUFFICIENT HISTORY", 0
    if c >= extreme_hi and m <= extreme_lo:
        return "BULLISH — commercials long, specs washed out", +1
    if c <= extreme_lo and m >= extreme_hi:
        return "BEARISH — commercials short, specs crowded", -1
    return f"NEUTRAL — no extreme (comm {c:.0f}, mm {m:.0f})", 0


Gold Smart Money Flow Intelligence
¶
Three signals: GLD ETF Flows | COT Swap Dealers | GLD Options Unusual Activity
¶
Run all cells top to bottom. Requires: 
pip install yfinance pandas requests matplotlib seaborn

In [ ]:
# ── INSTALL (run once) ──────────────────────────────────────────────────────
# !pip install yfinance pandas requests matplotlib seaborn


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a1a'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['grid.color']       = '#2a2a2a'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['font.size']        = 11

GOLD_COLOR   = '#FFD700'
GREEN_COLOR  = '#00C896'
RED_COLOR    = '#FF4D4D'
BLUE_COLOR   = '#4D9FFF'
ORANGE_COLOR = '#FF8C42'

print('Libraries loaded OK')


SIGNAL 1 — GLD ETF Institutional Flows
¶
What it tells you:
 GLD shares outstanding change daily as Authorized Participants (Goldman, JPMorgan, etc.) create or redeem baskets.




Shares outstanding 
rising
 = institutions accumulating gold → bullish


Shares outstanding 
falling
 = institutions distributing gold → bearish


Big divergence (price up, flows down) = distribution into strength → warning signal

In [ ]:
# ── GLD ETF FLOWS ───────────────────────────────────────────────────────────
LOOKBACK = '1y'   # change to '6mo', '2y' etc.

gld  = yf.Ticker('GLD')
gold = yf.Ticker('GC=F')

gld_hist  = gld.history(period=LOOKBACK)[['Close', 'Volume']]
gold_hist = gold.history(period=LOOKBACK)[['Close']]
gold_hist.columns = ['Gold_Futures']

# Shares outstanding proxy via volume accumulation (On-Balance Volume)
gld_hist['OBV'] = (np.sign(gld_hist['Close'].diff()) * gld_hist['Volume']).cumsum()
gld_hist['OBV_MA20'] = gld_hist['OBV'].rolling(20).mean()

# 5-day and 20-day flow momentum
gld_hist['Flow_5d']  = gld_hist['Volume'].rolling(5).mean()
gld_hist['Flow_20d'] = gld_hist['Volume'].rolling(20).mean()
gld_hist['Flow_Signal'] = gld_hist['Flow_5d'] / gld_hist['Flow_20d']  # >1 = above avg flow

df = gld_hist.join(gold_hist, how='inner').dropna()

# ── PLOT ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('SIGNAL 1 — GLD ETF Institutional Flows', fontsize=14, color=GOLD_COLOR, y=0.98)

# Panel 1: Gold price
axes[0].plot(df.index, df['Gold_Futures'], color=GOLD_COLOR, linewidth=1.5, label='Gold Futures')
axes[0].set_ylabel('Price (USD)')
axes[0].set_title('Gold Futures Price', fontsize=10)
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True)

# Panel 2: OBV (On-Balance Volume) — accumulation vs distribution
axes[1].plot(df.index, df['OBV'], color=BLUE_COLOR, linewidth=1.2, label='OBV (Cumulative Flow)')
axes[1].plot(df.index, df['OBV_MA20'], color=ORANGE_COLOR, linewidth=1, linestyle='--', label='OBV 20-day MA')
axes[1].fill_between(df.index,
                     df['OBV'], df['OBV_MA20'],
                     where=(df['OBV'] > df['OBV_MA20']),
                     alpha=0.15, color=GREEN_COLOR, label='Accumulation')
axes[1].fill_between(df.index,
                     df['OBV'], df['OBV_MA20'],
                     where=(df['OBV'] < df['OBV_MA20']),
                     alpha=0.15, color=RED_COLOR, label='Distribution')
axes[1].set_ylabel('OBV')
axes[1].set_title('On-Balance Volume — Institutional Accumulation / Distribution', fontsize=10)
axes[1].legend(loc='upper left', fontsize=9)
axes[1].grid(True)

# Panel 3: Flow momentum signal
colors = [GREEN_COLOR if v >= 1 else RED_COLOR for v in df['Flow_Signal']]
axes[2].bar(df.index, df['Flow_Signal'] - 1, color=colors, alpha=0.7, width=1)
axes[2].axhline(0, color='white', linewidth=0.8, linestyle='--')
axes[2].set_ylabel('Flow Ratio (5d/20d)')
axes[2].set_title('Flow Momentum — Above 0 = Unusual Buying Activity', fontsize=10)
axes[2].grid(True)

plt.tight_layout()
plt.show()

# ── SIGNAL SUMMARY ──────────────────────────────────────────────────────────
latest = df.iloc[-1]
obv_trend = 'ACCUMULATION' if latest['OBV'] > latest['OBV_MA20'] else 'DISTRIBUTION'
flow_mom  = 'ABOVE AVG' if latest['Flow_Signal'] > 1 else 'BELOW AVG'
obv_color = GREEN_COLOR if obv_trend == 'ACCUMULATION' else RED_COLOR

print('='*55)
print('SIGNAL 1 — GLD ETF FLOW READING')
print('='*55)
print(f'OBV Trend:      {obv_trend}')
print(f'Flow Momentum:  {flow_mom}  ({latest["Flow_Signal"]:.2f}x average)')
print(f'Gold Price:     ${latest["Gold_Futures"]:,.0f}')
print('='*55)
if obv_trend == 'ACCUMULATION' and flow_mom == 'ABOVE AVG':
    print('>>> SIGNAL: INSTITUTIONS ACTIVELY BUYING — BULLISH')
elif obv_trend == 'DISTRIBUTION' and flow_mom == 'BELOW AVG':
    print('>>> SIGNAL: INSTITUTIONS SELLING INTO STRENGTH — BEARISH')
else:
    print('>>> SIGNAL: MIXED / NEUTRAL — WAIT FOR CONFIRMATION')
print('='*55)


SIGNAL 2 — COT Swap Dealers (Bank Proxy)
¶
What it tells you:
 The CFTC Disaggregated COT report breaks out Swap Dealers separately from Managed Money.




Swap Dealers
 = banks and dealers hedging OTC gold exposure. This IS the London market proxy.


When Swap Dealers are 
net long
 → banks are accumulating physical/OTC exposure → bullish


When Swap Dealers are 
net short
 at extremes → commercials hedging a top → bearish warning


Divergence
 between Swap Dealers and Managed Money = the best contrarian signal in gold

In [ ]:
# ── COT DISAGGREGATED DATA ───────────────────────────────────────────────────
import io, zipfile, requests
import pandas as pd
from datetime import datetime

current_year = datetime.now().year

def load_cot_zip(url):
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        all_files = z.namelist()
        print(f'  Files in zip: {all_files}')

        # Prefer CSV; fall back to XLS/XLSX
        csv_files = [f for f in all_files if f.lower().endswith('.csv')]
        xls_files = [f for f in all_files if f.lower().endswith('.xls') or f.lower().endswith('.xlsx')]

        if csv_files:
            fname = csv_files[0]
            with z.open(fname) as f:
                # Try common encodings for CFTC files
                raw_bytes = f.read()
                for enc in ('utf-8', 'latin-1', 'cp1252'):
                    try:
                        return pd.read_csv(io.BytesIO(raw_bytes), encoding=enc, low_memory=False)
                    except UnicodeDecodeError:
                        continue
                raise ValueError(f'Could not decode {fname} with any known encoding')
        elif xls_files:
            fname = xls_files[0]
            print(f'  No CSV found, reading Excel file: {fname}')
            with z.open(fname) as f:
                return pd.read_excel(f)
        else:
            raise FileNotFoundError(f'No CSV or XLS file found in zip. Contents: {all_files}')

print(f'Downloading COT data for {current_year}...')
try:
    url = f'https://www.cftc.gov/files/dea/history/fut_disagg_xls_{current_year}.zip'
    cot_raw = load_cot_zip(url)
    print(f'Loaded current year data: {len(cot_raw)} rows')
except Exception as e:
    print(f'Download failed: {e}')
    print('Trying prior year...')
    url = f'https://www.cftc.gov/files/dea/history/fut_disagg_xls_{current_year - 1}.zip'
    cot_raw = load_cot_zip(url)
    print(f'Loaded prior year data: {len(cot_raw)} rows')


In [ ]:
# ── FILTER GOLD & EXTRACT KEY COLUMNS ───────────────────────────────────────
gold_cot = cot_raw[cot_raw['CFTC_Contract_Market_Code'] == '088691'].copy()

if len(gold_cot) == 0:
    # Try name match
    gold_cot = cot_raw[cot_raw['Market_and_Exchange_Names'].str.contains('GOLD', na=False, case=False)].copy()

print(f'Gold COT rows found: {len(gold_cot)}')

gold_cot['Report_Date'] = pd.to_datetime(gold_cot['Report_Date_as_MM_DD_YYYY'], errors='coerce')
gold_cot = gold_cot.sort_values('Report_Date').reset_index(drop=True)

# Key columns
cols = {
    'Report_Date': 'Date',
    'Swap_Positions_Long_All':  'Swap_Long',
    'Swap__Positions_Short_All': 'Swap_Short',
    'M_Money_Positions_Long_All': 'MM_Long',
    'M_Money_Positions_Short_All': 'MM_Short',
    'Prod_Merc_Positions_Long_All': 'Prod_Long',
    'Prod_Merc_Positions_Short_All': 'Prod_Short',
    'Open_Interest_All': 'Open_Interest'
}

available = {k: v for k, v in cols.items() if k in gold_cot.columns}
cot = gold_cot[list(available.keys())].rename(columns=available).copy()
cot = cot.dropna(subset=['Date']).set_index('Date')

for col in ['Swap_Long','Swap_Short','MM_Long','MM_Short']:
    if col in cot.columns:
        cot[col] = pd.to_numeric(cot[col], errors='coerce')

if 'Swap_Long' in cot.columns and 'Swap_Short' in cot.columns:
    cot['Swap_Net']  = cot['Swap_Long']  - cot['Swap_Short']
if 'MM_Long' in cot.columns and 'MM_Short' in cot.columns:
    cot['MM_Net']    = cot['MM_Long']    - cot['MM_Short']

cot = cot.tail(52)  # last 52 weeks

# ── PLOT ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('SIGNAL 2 — COT Swap Dealers vs Managed Money (Bank Proxy)', fontsize=14, color=GOLD_COLOR, y=0.98)

# Panel 1: Swap Dealers Net
if 'Swap_Net' in cot.columns:
    colors_swap = [GREEN_COLOR if v >= 0 else RED_COLOR for v in cot['Swap_Net']]
    axes[0].bar(cot.index, cot['Swap_Net'], color=colors_swap, alpha=0.75, width=5)
    axes[0].axhline(0, color='white', linewidth=0.8, linestyle='--')
    axes[0].set_title('Swap Dealers Net Position (Bank/London Proxy) — Contracts', fontsize=10)
    axes[0].set_ylabel('Net Contracts')
    axes[0].grid(True)

# Panel 2: Managed Money Net
if 'MM_Net' in cot.columns:
    colors_mm = [GREEN_COLOR if v >= 0 else RED_COLOR for v in cot['MM_Net']]
    axes[1].bar(cot.index, cot['MM_Net'], color=colors_mm, alpha=0.75, width=5)
    axes[1].axhline(0, color='white', linewidth=0.8, linestyle='--')
    axes[1].set_title('Managed Money Net Position (Hedge Funds / Speculators) — Contracts', fontsize=10)
    axes[1].set_ylabel('Net Contracts')
    axes[1].grid(True)

# Panel 3: Divergence (when they disagree = opportunity)
if 'Swap_Net' in cot.columns and 'MM_Net' in cot.columns:
    cot['Divergence'] = cot['Swap_Net'] - cot['MM_Net']
    div_colors = [BLUE_COLOR if v >= 0 else ORANGE_COLOR for v in cot['Divergence']]
    axes[2].bar(cot.index, cot['Divergence'], color=div_colors, alpha=0.75, width=5)
    axes[2].axhline(0, color='white', linewidth=0.8, linestyle='--')
    axes[2].set_title('Divergence: Swap Dealers MINUS Managed Money — Blue=Banks more bullish than funds', fontsize=10)
    axes[2].set_ylabel('Divergence')
    axes[2].grid(True)

plt.tight_layout()
plt.show()

# ── SIGNAL SUMMARY ──────────────────────────────────────────────────────────
latest_cot = cot.iloc[-1]
print('='*55)
print('SIGNAL 2 — COT SWAP DEALER READING (latest week)')
print('='*55)
if 'Swap_Net' in cot.columns:
    swap_pct = (cot['Swap_Net'].rank(pct=True).iloc[-1]) * 100
    mm_pct   = (cot['MM_Net'].rank(pct=True).iloc[-1]) * 100  if 'MM_Net' in cot.columns else None
    print(f'Swap Dealer Net:   {int(latest_cot["Swap_Net"]):,} contracts  ({swap_pct:.0f}th pct of 1yr)')
    if mm_pct: print(f'Managed Money Net: {int(latest_cot["MM_Net"]):,} contracts  ({mm_pct:.0f}th pct of 1yr)')
    print('='*55)
    if swap_pct > 70:
        print('>>> SIGNAL: BANKS NET LONG — BULLISH BIAS')
    elif swap_pct < 30:
        print('>>> SIGNAL: BANKS NET SHORT — BEARISH BIAS')
    else:
        print('>>> SIGNAL: NEUTRAL — NO EXTREME READING')
    if mm_pct and swap_pct > 60 and mm_pct < 40:
        print('>>> DIVERGENCE ALERT: Banks bullish while funds bearish — Strong contrarian BUY')
    elif mm_pct and swap_pct < 40 and mm_pct > 60:
        print('>>> DIVERGENCE ALERT: Banks bearish while funds bullish — Strong contrarian SELL')
print('='*55)


SIGNAL 3 — GLD Options Unusual Activity
¶
What it tells you:
 When large institutions want to take a directional gold position quietly, they often use options. The tell is when options 
volume
 far exceeds 
open interest
 on a specific strike — that means fresh, large positions being opened.




Volume/OI ratio > 2 on calls
 = unusual call buying → bullish bet


Volume/OI ratio > 2 on puts
 = unusual put buying → bearish bet / hedge


The 
put/call ratio
 tells you the overall skew of institutional hedging

In [ ]:
# ── GLD OPTIONS UNUSUAL ACTIVITY ────────────────────────────────────────────
gld = yf.Ticker('GLD')

# Get available expiry dates
expiries = gld.options
print(f'Available GLD expiry dates ({len(expiries)} total):')
for i, exp in enumerate(expiries[:8]):
    print(f'  [{i}] {exp}')
print()

# ── Use nearest 3 expiries for best signal ───────────────────────────────────
all_calls, all_puts = [], []

for exp in expiries[:3]:
    chain = gld.option_chain(exp)
    calls = chain.calls.copy()
    puts  = chain.puts.copy()
    calls['expiry'] = exp
    puts['expiry']  = exp
    all_calls.append(calls)
    all_puts.append(puts)

calls_df = pd.concat(all_calls, ignore_index=True)
puts_df  = pd.concat(all_puts,  ignore_index=True)

# Clean numeric
for col in ['volume', 'openInterest', 'lastPrice', 'impliedVolatility', 'strike']:
    calls_df[col] = pd.to_numeric(calls_df[col], errors='coerce').fillna(0)
    puts_df[col]  = pd.to_numeric(puts_df[col],  errors='coerce').fillna(0)

# Volume/OI ratio — the unusual activity signal
calls_df['Vol_OI_Ratio'] = calls_df['volume'] / (calls_df['openInterest'] + 1)
puts_df['Vol_OI_Ratio']  = puts_df['volume']  / (puts_df['openInterest']  + 1)

# Filter meaningful activity
unusual_calls = calls_df[(calls_df['volume'] > 100) & (calls_df['Vol_OI_Ratio'] > 1.5)].sort_values('volume', ascending=False).head(10)
unusual_puts  = puts_df[ (puts_df['volume']  > 100) & (puts_df['Vol_OI_Ratio']  > 1.5)].sort_values('volume', ascending=False).head(10)

# ── PLOT ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SIGNAL 3 — GLD Options Unusual Activity (Smart Money)', fontsize=14, color=GOLD_COLOR, y=0.98)

spot = gld.history(period='1d')['Close'].iloc[-1]

# Top-left: Unusual calls by strike
if len(unusual_calls) > 0:
    bars = axes[0,0].barh(unusual_calls['strike'].astype(str) + ' (' + unusual_calls['expiry'] + ')',
                          unusual_calls['volume'], color=GREEN_COLOR, alpha=0.75)
    axes[0,0].set_title('Unusual CALL Activity (Bullish Bets)', fontsize=10, color=GREEN_COLOR)
    axes[0,0].set_xlabel('Volume')
    axes[0,0].grid(True, axis='x')

# Top-right: Unusual puts by strike
if len(unusual_puts) > 0:
    axes[0,1].barh(unusual_puts['strike'].astype(str) + ' (' + unusual_puts['expiry'] + ')',
                   unusual_puts['volume'], color=RED_COLOR, alpha=0.75)
    axes[0,1].set_title('Unusual PUT Activity (Bearish Bets / Hedges)', fontsize=10, color=RED_COLOR)
    axes[0,1].set_xlabel('Volume')
    axes[0,1].grid(True, axis='x')

# Bottom-left: IV skew (put IV vs call IV by strike)
near_calls = calls_df[(calls_df['openInterest'] > 50)].copy()
near_puts  = puts_df[ (puts_df['openInterest']  > 50)].copy()
if len(near_calls) > 0 and len(near_puts) > 0:
    axes[1,0].plot(near_calls['strike'], near_calls['impliedVolatility'], color=GREEN_COLOR, linewidth=1.2, label='Call IV')
    axes[1,0].plot(near_puts['strike'],  near_puts['impliedVolatility'],  color=RED_COLOR,   linewidth=1.2, label='Put IV')
    axes[1,0].axvline(spot, color=GOLD_COLOR, linewidth=1.5, linestyle='--', label=f'Spot ${spot:.0f}')
    axes[1,0].set_title('Implied Volatility Skew — Put IV > Call IV = Fear', fontsize=10)
    axes[1,0].set_xlabel('Strike')
    axes[1,0].set_ylabel('Implied Volatility')
    axes[1,0].legend(fontsize=9)
    axes[1,0].grid(True)

# Bottom-right: Put/Call ratio by expiry
pc_data = []
for exp in expiries[:5]:
    chain = gld.option_chain(exp)
    c_vol = pd.to_numeric(chain.calls['volume'], errors='coerce').fillna(0).sum()
    p_vol = pd.to_numeric(chain.puts['volume'],  errors='coerce').fillna(0).sum()
    if c_vol > 0:
        pc_data.append({'expiry': exp, 'PC_Ratio': p_vol / c_vol, 'Call_Vol': c_vol, 'Put_Vol': p_vol})

pc_df = pd.DataFrame(pc_data)
if len(pc_df) > 0:
    pc_colors = [RED_COLOR if r > 1 else GREEN_COLOR for r in pc_df['PC_Ratio']]
    axes[1,1].bar(pc_df['expiry'], pc_df['PC_Ratio'], color=pc_colors, alpha=0.75)
    axes[1,1].axhline(1.0, color='white', linewidth=0.8, linestyle='--', label='Neutral (1.0)')
    axes[1,1].set_title('Put/Call Volume Ratio by Expiry — >1 = More Put Buying (Fear)', fontsize=10)
    axes[1,1].set_xlabel('Expiry')
    axes[1,1].set_ylabel('P/C Ratio')
    axes[1,1].tick_params(axis='x', rotation=30)
    axes[1,1].legend(fontsize=9)
    axes[1,1].grid(True, axis='y')

plt.tight_layout()
plt.show()

# ── SIGNAL SUMMARY ──────────────────────────────────────────────────────────
total_call_vol = calls_df['volume'].sum()
total_put_vol  = puts_df['volume'].sum()
overall_pc     = total_put_vol / (total_call_vol + 1)

print('='*55)
print('SIGNAL 3 — GLD OPTIONS UNUSUAL ACTIVITY READING')
print('='*55)
print(f'GLD Spot:            ${spot:.2f}')
print(f'Total Call Volume:   {int(total_call_vol):,}')
print(f'Total Put Volume:    {int(total_put_vol):,}')
print(f'Overall P/C Ratio:   {overall_pc:.2f}')
print(f'Unusual Call Strikes (top 3):')
for _, row in unusual_calls.head(3).iterrows():
    print(f'  Strike ${row["strike"]:.0f}  Vol={int(row["volume"]):,}  Vol/OI={row["Vol_OI_Ratio"]:.1f}x  Exp={row["expiry"]}')
print(f'Unusual Put Strikes (top 3):')
for _, row in unusual_puts.head(3).iterrows():
    print(f'  Strike ${row["strike"]:.0f}  Vol={int(row["volume"]):,}  Vol/OI={row["Vol_OI_Ratio"]:.1f}x  Exp={row["expiry"]}')
print('='*55)
if overall_pc < 0.7:
    print('>>> SIGNAL: HEAVY CALL SKEW — INSTITUTIONS POSITIONED BULLISH')
elif overall_pc > 1.3:
    print('>>> SIGNAL: HEAVY PUT SKEW — INSTITUTIONS HEDGING / BEARISH')
else:
    print('>>> SIGNAL: BALANCED P/C — NO STRONG DIRECTIONAL BIAS')
print('='*55)


FINAL SCORECARD — All 3 Signals Combined
¶

In [ ]:
# ── COMBINED SCORECARD ───────────────────────────────────────────────────────
# Scores: +1 = Bullish, 0 = Neutral, -1 = Bearish

scores = {}

# Signal 1: ETF Flows
latest_gld = df.iloc[-1]
if latest_gld['OBV'] > latest_gld['OBV_MA20'] and latest_gld['Flow_Signal'] > 1:
    scores['GLD ETF Flows'] = (+1, 'Accumulation + Above-avg flow', GREEN_COLOR)
elif latest_gld['OBV'] < latest_gld['OBV_MA20'] and latest_gld['Flow_Signal'] < 1:
    scores['GLD ETF Flows'] = (-1, 'Distribution + Below-avg flow', RED_COLOR)
else:
    scores['GLD ETF Flows'] = (0, 'Mixed signals', ORANGE_COLOR)

# Signal 2: COT Swap Dealers
if 'Swap_Net' in cot.columns:
    swap_pct_final = cot['Swap_Net'].rank(pct=True).iloc[-1] * 100
    if swap_pct_final > 65:
        scores['COT Swap Dealers'] = (+1, f'Banks net long  ({swap_pct_final:.0f}th pct)', GREEN_COLOR)
    elif swap_pct_final < 35:
        scores['COT Swap Dealers'] = (-1, f'Banks net short ({swap_pct_final:.0f}th pct)', RED_COLOR)
    else:
        scores['COT Swap Dealers'] = (0, f'Neutral ({swap_pct_final:.0f}th pct)', ORANGE_COLOR)

# Signal 3: Options P/C
if overall_pc < 0.7:
    scores['GLD Options'] = (+1, f'Call skew P/C={overall_pc:.2f} — Bullish positioning', GREEN_COLOR)
elif overall_pc > 1.3:
    scores['GLD Options'] = (-1, f'Put skew P/C={overall_pc:.2f} — Bearish/hedging', RED_COLOR)
else:
    scores['GLD Options'] = (0, f'Neutral P/C={overall_pc:.2f}', ORANGE_COLOR)

# ── PRINT SCORECARD ─────────────────────────────────────────────────────────
total = sum(v[0] for v in scores.values())

print()
print('=' * 60)
print('  GOLD SMART MONEY SCORECARD')
print('=' * 60)
for signal, (score, detail, _) in scores.items():
    arrow = '+1 BULL' if score == 1 else ('-1 BEAR' if score == -1 else ' 0 NEUT')
    print(f'  [{arrow}]  {signal}')
    print(f'           {detail}')
    print()
print('-' * 60)
print(f'  COMPOSITE SCORE: {total:+d} out of +{len(scores)}')
print()
if total >= 2:
    verdict = 'SMART MONEY BULLISH — Align with long bias'
elif total <= -2:
    verdict = 'SMART MONEY BEARISH — Align with short bias'
elif total == 1:
    verdict = 'SLIGHT BULL LEAN — Confirm with technicals'
elif total == -1:
    verdict = 'SLIGHT BEAR LEAN — Confirm with technicals'
else:
    verdict = 'NO CLEAR EDGE — Stay small, wait for clarity'
print(f'  VERDICT: {verdict}')
print('=' * 60)
